# ⚙️ Notebook 02 — Preprocessing & Feature Engineering Pipeline
### Amazon Fine Food Reviews · IASD Big Data Project 2025-2026

---

## 🎯 Goal

Build, validate, and save a **PySpark ML preprocessing pipeline** that transforms  
raw Amazon review text into feature vectors ready for classification.

| Step | Action | Output |
|------|--------|--------|
| **1** | Load & deduplicate | Clean DataFrame |
| **2** | Label creation | `label` column (0/1/2) |
| **3** | Text cleaning (regex) | `cleaned_text` column |
| **4** | Feature engineering | Unigram + Bigram TF-IDF vectors |
| **5** | Validation tests | Proof each step works correctly |
| **6** | 80 / 10 / 10 split | Train, Validation, Test DataFrames |
| **7** | Save pipeline | `models/preprocessing_pipeline` |

---

> **Design principle:**  
> This pipeline is **fitted on training data only**.  
> It is then applied (`.transform()`) to validation and test sets  
> without refitting — this prevents data leakage.


In [6]:
# ============================================================
# SECTION 1 — Spark Session
# ============================================================

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, when, lower, regexp_replace,
    concat_ws, coalesce, lit, length, udf, count
)
from pyspark.sql.types import IntegerType
from pyspark.ml.feature import (
    Tokenizer, StopWordsRemover,
    NGram, CountVectorizer, IDF,
    StringIndexer, VectorAssembler
)
from pyspark.ml import Pipeline
import re
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

spark = SparkSession.builder \
    .appName("02_Preprocessing_Pipeline") \
    .config("spark.driver.memory", "10g") \
    .config("spark.sql.shuffle.partitions", "200") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"✅ Spark {spark.version} ready.")


✅ Spark 4.0.2 ready.


In [7]:
# ============================================================
# SECTION 2 — Load, Deduplicate & Label Creation
# ============================================================

# ── 2.1 Robust score parser (handles edge cases) ────────────
def safe_int_cast(s):
    """Parse Score robustly — handles nulls, floats, stray characters."""
    try:
        if s is None:
            return None
        cleaned = re.sub(r"[^0-9]", "", str(s))
        return int(cleaned) if cleaned else None
    except Exception:
        return None

safe_int_udf = udf(safe_int_cast, IntegerType())

# ── 2.2 Load CSV ────────────────────────────────────────────
df = spark.read.csv(
    "Reviews.csv",
    header      = True,
    inferSchema = True,
    multiLine   = True,
    quote       = '"',
    escape      = '"'
)

df = df.withColumnRenamed("Summary", "Review_Summary")
df = df.withColumn("Score", safe_int_udf(col("Score")))

# Drop rows with null Score or Text (EDA showed 0 nulls, but defensive check)
df = df.na.drop(subset=["Score", "Text"])

# Remove duplicate reviews (same text) — prevents train/test leakage
before_dedup = df.count()
df = df.dropDuplicates(["Text"])
after_dedup  = df.count()
print(f"Rows before dedup : {before_dedup:,}")
print(f"Rows after  dedup : {after_dedup:,}  (removed {before_dedup-after_dedup:,} duplicates)")

# ── 2.3 Create target label ─────────────────────────────────
# Score < 3  → 0 (Negative)
# Score == 3 → 1 (Neutral)
# Score > 3  → 2 (Positive)
df = df.withColumn("label",
    when(col("Score") < 3,  0)
    .when(col("Score") == 3, 1)
    .otherwise(              2)
)

# Confirm label distribution
print("\nLabel distribution after dedup:")
df.groupBy("label").count().orderBy("label").show()


Rows before dedup : 568,454
Rows after  dedup : 393,579  (removed 174,875 duplicates)

Label distribution after dedup:
+-----+------+
|label| count|
+-----+------+
|    0| 57067|
|    1| 29754|
|    2|306758|
+-----+------+



In [8]:
# ============================================================
# SECTION 3 — Text Cleaning
# ============================================================
# EDA Finding #7: Combine Summary + Text for richer signal.
# EDA Finding #4: Remove HTML tags present in raw data.
# EDA Finding #5: Keep full text — do NOT truncate.

# ── 3.1 Combine Summary + Text ──────────────────────────────
# Using coalesce to handle null Summary gracefully
df = df.withColumn(
    "full_text",
    concat_ws(
        " ",
        coalesce(col("Review_Summary"), lit("")),
        coalesce(col("Text"),           lit(""))
    )
)

# ── 3.2 Apply cleaning pipeline ────────────────────────────
# Step 1: Lowercase
df = df.withColumn("cleaned_text", lower(col("full_text")))

# Step 2: Remove HTML tags  (e.g. <br />, <b>, &amp;)
df = df.withColumn("cleaned_text",
    regexp_replace(col("cleaned_text"), r"<[^>]+>", " "))

# Step 3: Remove URLs
df = df.withColumn("cleaned_text",
    regexp_replace(col("cleaned_text"), r"https?://\S+|www\.\S+", " "))

# Step 4: Keep only alphabetic characters + spaces
df = df.withColumn("cleaned_text",
    regexp_replace(col("cleaned_text"), r"[^a-z\s]", " "))

# Step 5: Collapse multiple whitespace
df = df.withColumn("cleaned_text",
    regexp_replace(col("cleaned_text"), r"\s+", " "))

# Drop rows where cleaning produced empty or near-empty text
df = df.filter(length(col("cleaned_text")) > 10)
df = df.na.drop(subset=["cleaned_text"])

print(f"Rows after cleaning: {df.count():,}")

# ── 3.3 Visual validation ───────────────────────────────────
print("\n--- VALIDATION: Raw vs Cleaned (5 rows) ---")
df.select("full_text", "cleaned_text").show(5, truncate=80)

# Check: ensure no HTML artifacts remain
html_remaining = df.filter(col("cleaned_text").contains("<")).count()
print(f"\nRows with '<' after cleaning: {html_remaining}  (should be 0)")

# Check: ensure no URLs remain
url_remaining = df.filter(col("cleaned_text").rlike(r"http|www")).count()
print(f"Rows with URL artifacts: {url_remaining}  (should be 0)")


Rows after cleaning: 393,578

--- VALIDATION: Raw vs Cleaned (5 rows) ---
+--------------------------------------------------------------------------------+--------------------------------------------------------------------------------+
|                                                                       full_text|                                                                    cleaned_text|
+--------------------------------------------------------------------------------+--------------------------------------------------------------------------------+
|sugar ! think I just ruined my dish. open this and poured it in. some past st...|sugar think i just ruined my dish open this and poured it in some past stuck ...|
|!!! FALSE ADVERTISEMENT !!! !!! PRODUCT HAS NOTHING TO DO WITH MARVEL!!! FALS...| false advertisement product has nothing to do with marvel false advertisemen...|
|TWO big candy bars in EACH Package! !!HEY, these are BIG! There are actually ...|two big candy bars in ea

---
## ✂️ Section 4 — Strict 80 / 10 / 10 Data Split

**Why three splits and not two?**

A 80/20 (train/test) split is a common mistake in academic projects.  
When you use the test set to decide which hyperparameters to keep,  
you have implicitly trained on it — the test score is no longer unbiased.

| Split | Size | Purpose | When used |
|-------|------|---------|-----------|
| **Train**      | 80% | Fit all pipeline stages & model weights | Notebooks 02 and 03 |
| **Validation** | 10% | Tune hyperparameters (regParam, maxIter) | Notebook 03 only |
| **Test**       | 10% | Final unbiased evaluation | **Once** — end of Notebook 03 |

> **Stratified splits** are not natively supported in PySpark ML.  
> We implement manual stratification to preserve class proportions.


In [9]:
# ============================================================
# SECTION 4 — Stratified 80 / 10 / 10 Split
# ============================================================

def stratified_split_80_10_10(df, label_col="label", seed=42):
    """
    Perform stratified 80/10/10 split preserving class proportions.

    PySpark's randomSplit() is not stratified — on imbalanced datasets
    it can produce splits with very different class ratios, which makes
    validation metrics unreliable. We split per class then union.

    Parameters
    ----------
    df         : Spark DataFrame with a label column
    label_col  : name of the integer label column
    seed       : random seed for reproducibility

    Returns
    -------
    (train_df, val_df, test_df)
    """
    labels = [row[label_col] for row in df.select(label_col).distinct().collect()]
    train_parts, val_parts, test_parts = [], [], []

    for lbl in sorted(labels):
        subset = df.filter(col(label_col) == lbl)
        # First: 80% train | 20% temp
        tr, tmp = subset.randomSplit([0.80, 0.20], seed=seed)
        # Second: 50% of 20% = 10% val | 10% test
        vl, te  = tmp.randomSplit([0.50, 0.50], seed=seed)
        train_parts.append(tr)
        val_parts.append(vl)
        test_parts.append(te)

    train_df = train_parts[0]
    val_df   = val_parts[0]
    test_df  = test_parts[0]
    for i in range(1, len(labels)):
        train_df = train_df.union(train_parts[i])
        val_df   = val_df.union(val_parts[i])
        test_df  = test_df.union(test_parts[i])

    return train_df, val_df, test_df


# ── Apply stratified split ──────────────────────────────────
train_df, val_df, test_df = stratified_split_80_10_10(df, label_col="label")

# Cache the splits — they will be reused multiple times during training
train_df.cache()
val_df.cache()
test_df.cache()

# ── Verify split sizes and class proportions ────────────────
total = df.count()
splits = {"Train": train_df, "Validation": val_df, "Test": test_df}

print(f"{'Split':<12} {'N':>10}  {'%':>6}  {'Neg%':>7}  {'Neu%':>7}  {'Pos%':>7}")
print("-" * 58)
for name, sdf in splits.items():
    n     = sdf.count()
    neg   = sdf.filter(col("label") == 0).count()
    neu   = sdf.filter(col("label") == 1).count()
    pos   = sdf.filter(col("label") == 2).count()
    print(f"{name:<12} {n:>10,}  {n/total*100:>5.1f}%  "
          f"{neg/n*100:>6.1f}%  {neu/n*100:>6.1f}%  {pos/n*100:>6.1f}%")

print("\n✅ Class proportions consistent across all three splits.")
print("   (Variation < 0.5% per class — stratification successful)")


Split                 N       %     Neg%     Neu%     Pos%
----------------------------------------------------------
Train           315,004   80.0%    14.5%     7.5%    78.0%
Validation       39,230   10.0%    14.5%     7.6%    77.9%
Test             39,344   10.0%    14.6%     7.7%    77.7%

✅ Class proportions consistent across all three splits.
   (Variation < 0.5% per class — stratification successful)


---
## 🔧 Section 5 — Feature Engineering Pipeline

### Architecture

```
cleaned_text (string)
      │
      ▼
Tokenizer               → splits on whitespace → ["great", "product", ...]
      │
      ▼
StopWordsRemover        → removes "the", "is", "at", ... → ["great", "product", ...]
      │
      ├──────────────────────────────────────────┐
      ▼                                          ▼
PassThrough (unigrams)                     NGram(n=2) (bigrams)
["great","product",...]             ["great product","product quality",...]
      │                                          │
      └──────────────┬───────────────────────────┘
                     │  VectorAssembler combines both
                     ▼
           CountVectorizer (vocabSize=15,000)   → sparse raw TF vector
                     │
                     ▼
               IDF Transformer                  → TF-IDF weighted vector
                     │
                     ▼
             StringIndexer                      → label_idx (0.0/1.0/2.0)
```

### Why unigrams AND bigrams?

**EDA Finding #7:** Negation patterns like "not good", "not bad", "highly recommend"  
cannot be captured by unigrams alone. Bigrams add context that single words miss.

 We combine both using `VectorAssembler`.


In [10]:
# ============================================================
# SECTION 5 — Build Preprocessing Pipeline
# ============================================================
from pyspark.ml.feature import (
    Tokenizer, StopWordsRemover, NGram,
    CountVectorizer, IDF, StringIndexer, VectorAssembler
)
from pyspark.ml import Pipeline

# ── Stage 1: Tokenizer ────────────────────────────────────
# Splits on whitespace. Input: cleaned_text string.
tokenizer = Tokenizer(
    inputCol  = "cleaned_text",
    outputCol = "words_raw"
)

# ── Stage 2: StopWords Removal ────────────────────────────
# Removes English stopwords (the, is, at, which, ...)
# caseSensitive=False: words are already lowercased, but defensive setting
remover = StopWordsRemover(
    inputCol      = "words_raw",
    outputCol     = "words_filtered",
    caseSensitive = False
)

# ── Stage 3a: Bigram Extractor ────────────────────────────
# EDA Finding #7: captures "not good", "highly recommend", etc.
ngram = NGram(
    n         = 2,
    inputCol  = "words_filtered",
    outputCol = "bigrams"
)

# ── Stage 4a: CountVectorizer for UNIGRAMS ────────────────
# vocabSize=10000: top 10K unigrams by frequency
# minDF=5: ignore terms appearing in fewer than 5 documents
cv_unigram = CountVectorizer(
    inputCol  = "words_filtered",
    outputCol = "tf_unigram",
    vocabSize = 10_000,
    minDF     = 5
)

# ── Stage 4b: CountVectorizer for BIGRAMS ─────────────────
# vocabSize=10000: top 10K bigrams by frequency
cv_bigram = CountVectorizer(
    inputCol  = "bigrams",
    outputCol = "tf_bigram",
    vocabSize = 10_000,
    minDF     = 5
)

# ── Stage 5a: IDF for UNIGRAMS ────────────────────────────
# Downweights terms that appear in many documents (very common words)
idf_unigram = IDF(
    inputCol  = "tf_unigram",
    outputCol = "tfidf_unigram",
    minDocFreq = 5
)

# ── Stage 5b: IDF for BIGRAMS ─────────────────────────────
idf_bigram = IDF(
    inputCol  = "tf_bigram",
    outputCol = "tfidf_bigram",
    minDocFreq = 5
)

# ── Stage 6: Combine unigram + bigram vectors ─────────────
# VectorAssembler concatenates both sparse vectors into one feature vector
assembler = VectorAssembler(
    inputCols = ["tfidf_unigram", "tfidf_bigram"],
    outputCol = "features"
)

# ── Stage 7: Label Indexer ────────────────────────────────
# Converts integer label (0/1/2) to double (0.0/1.0/2.0) for PySpark ML
# stringOrderType="alphabetAsc" ensures: 0→0.0, 1→1.0, 2→2.0 (stable mapping)
label_indexer = StringIndexer(
    inputCol        = "label",
    outputCol       = "label_idx",
    stringOrderType = "alphabetAsc"
)

# ── Assemble the pipeline ─────────────────────────────────
preprocessing_stages = [
    tokenizer,      # split text into word tokens
    remover,        # remove stopwords
    ngram,          # extract bigrams
    cv_unigram,     # count unigrams  (fitted on train only)
    cv_bigram,      # count bigrams   (fitted on train only)
    idf_unigram,    # compute unigram TF-IDF
    idf_bigram,     # compute bigram  TF-IDF
    assembler,      # combine into single feature vector
    label_indexer,  # encode label
]

preprocessing_pipeline = Pipeline(stages=preprocessing_stages)
print("Pipeline defined with stages:")
for i, stage in enumerate(preprocessing_stages):
    print(f"  [{i}] {stage.__class__.__name__}")


Pipeline defined with stages:
  [0] Tokenizer
  [1] StopWordsRemover
  [2] NGram
  [3] CountVectorizer
  [4] CountVectorizer
  [5] IDF
  [6] IDF
  [7] VectorAssembler
  [8] StringIndexer


In [13]:
# ============================================================
# SECTION 6 — Fit Pipeline on Training Data & Validate
# ============================================================
# CRITICAL: Pipeline is fitted on TRAINING data only.
# This prevents vocabulary/IDF statistics from being contaminated
# by validation or test set information.

import time
from pyspark.sql.functions import size as spark_size, udf
from pyspark.sql.types import BooleanType
from pyspark.ml.linalg import SparseVector, DenseVector
from pyspark.sql.functions import col # Import col for clarity

# UDF to check if a vector is empty (all zeros)
@udf(BooleanType())
def is_vector_empty(vec):
    if vec is None:
        return True  # Consider null vectors as empty for this test
    if isinstance(vec, SparseVector):
        return vec.numNonzeros == 0
    elif isinstance(vec, DenseVector):
        return all(x == 0 for x in vec.values) # For DenseVector, check if all values are zero
    return False

print(f"Fitting du pipeline sur {train_df.count():,} exemples d'entraînement...")
t0 = time.time()

preprocessing_model = preprocessing_pipeline.fit(train_df)

print(f"✅ Pipeline fitté en {time.time()-t0:.1f}s")

# --- Transformation des 3 splits ---
print("\nTransformation Train / Validation / Test...")
t0 = time.time()

train_transformed = preprocessing_model.transform(train_df)
val_transformed   = preprocessing_model.transform(val_df)
test_transformed  = preprocessing_model.transform(test_df)

train_transformed.cache()
val_transformed.cache()
test_transformed.cache()

print(f"✅ Transformations en {time.time()-t0:.1f}s")

# ============================================================
# TESTS DE VALIDATION DU PIPELINE
# ============================================================

print("\n" + "="*60)
print("  TESTS DE VALIDATION")
print("="*60)

# TEST 1 : Dimension du vecteur features
sample_row  = train_transformed.select("features").first()
feature_dim = sample_row["features"].size
print(f"\n[TEST 1] Dimension vecteur features : {feature_dim:,}")
print(f"         Attendu : ~20,000 (10K unigrams + 10K bigrams)")
assert 15_000 <= feature_dim <= 25_000, f"Dimension inattendue : {feature_dim}"
print("         ✅ PASS")

# TEST 2 : Qualité du vocabulaire
vocab_uni = preprocessing_model.stages[3].vocabulary   # cv_unigram = index 3
vocab_bi  = preprocessing_model.stages[4].vocabulary   # cv_bigram  = index 4
print(f"\n[TEST 2] Tailles vocabulaires :")
print(f"         Unigrams : {len(vocab_uni):,}")
print(f"         Bigrams  : {len(vocab_bi):,}")
print(f"         Top 10 unigrams : {vocab_uni[:10]}")
print(f"         Top 10 bigrams  : {vocab_bi[:10]}")

bad = [w for w in vocab_uni[:200] if "br" in w or "http" in w or "<" in w]
print(f"         Artefacts HTML/URL dans unigrams : {bad or 'Aucun ✅'}")

# TEST 3 : Mapping des labels
print(f"\n[TEST 3] Mapping label → label_idx :")
train_transformed.select("label", "label_idx").distinct().orderBy("label").show()
print("         Attendu : 0→0.0, 1→1.0, 2→2.0")

# TEST 4 : Exemples de bigrams
print(f"[TEST 4] Exemples de bigrams (2 lignes) :")
train_transformed.select("bigrams").limit(2).show(truncate=100)

# TEST 5 : Aucun vecteur vide
# PySpark SparseVector est un UDT — on ne peut pas utiliser size() directement.
# On accède à l'attribut interne .values (ARRAY<DOUBLE>) du vecteur sparse,
# puis on vérifie que le nombre de valeurs non-nulles est > 0.
from pyspark.ml.linalg import VectorUDT
from pyspark.sql.functions import udf
from pyspark.sql.types import IntegerType

def vector_nnz(v):
    """Retourne le nombre de valeurs non-zéro d'un SparseVector."""
    if v is None:
        return 0
    return int(v.numNonzeros())

vector_nnz_udf = udf(vector_nnz, IntegerType())

empty_features = train_transformed.filter(
    vector_nnz_udf(col("features")) == 0
).count()
print(f"[TEST 5] Vecteurs features vides (nnz == 0) : {empty_features}  (attendu : 0)")
print(f"         {'✅ PASS' if empty_features == 0 else '❌ FAIL — investiguer !'}")



Fitting du pipeline sur 315,004 exemples d'entraînement...
✅ Pipeline fitté en 290.7s

Transformation Train / Validation / Test...
✅ Transformations en 1.0s

  TESTS DE VALIDATION

[TEST 1] Dimension vecteur features : 20,000
         Attendu : ~20,000 (10K unigrams + 10K bigrams)
         ✅ PASS

[TEST 2] Tailles vocabulaires :
         Unigrams : 10,000
         Bigrams  : 10,000
         Top 10 unigrams : ['like', 'good', 'great', 'taste', 'product', 'one', 'coffee', 'flavor', 'tea', 'love']
         Top 10 bigrams  : ['gluten free', 'great product', 'taste like', 'peanut butter', 'highly recommend', 've tried', 'green tea', 'grocery store', 'tastes like', 'much better']
         Artefacts HTML/URL dans unigrams : ['brand']

[TEST 3] Mapping label → label_idx :
+-----+---------+
|label|label_idx|
+-----+---------+
|    0|      0.0|
|    1|      1.0|
|    2|      2.0|
+-----+---------+

         Attendu : 0→0.0, 1→1.0, 2→2.0
[TEST 4] Exemples de bigrams (2 lignes) :
+----------------

In [15]:
# ============================================================
# SECTION 7 — Save Preprocessing Pipeline & Transformed Data
# ============================================================

import os

# Create model directory
os.makedirs("models", exist_ok=True)

# ── Save the fitted preprocessing pipeline ─────────────────
# This includes all fitted vocabularies and IDF statistics.
# The Spark Streaming job will load this to transform incoming reviews.
preprocessing_model.write().overwrite().save("models/preprocessing_pipeline")
print("✅ Preprocessing pipeline saved: models/preprocessing_pipeline")

# ── Save transformed splits as Parquet ────────────────────
# Parquet is columnar and much faster to reload than CSV.
# Notebook 03 will load these directly — no need to re-run the pipeline.
train_transformed.write.mode("overwrite").parquet("data/train_transformed")
val_transformed.write.mode("overwrite").parquet("data/val_transformed")
test_transformed.write.mode("overwrite").parquet("data/test_transformed")

print("✅ Transformed splits saved:")
print("   data/train_transformed/")
print("   data/val_transformed/")
print("   data/test_transformed/")
print()
print("="*55)
print("  HANDOVER SUMMARY")
print("="*55)
print("  Notebook 03 needs:")
print("  1. models/preprocessing_pipeline  ← fitted pipeline")
print("  2. data/train_transformed/        ← features + label_idx")
print("  3. data/val_transformed/          ← for hyperparameter tuning")
print("  4. data/test_transformed/         ← LOCKED until final eval")
print("="*55)


✅ Preprocessing pipeline saved: models/preprocessing_pipeline
✅ Transformed splits saved:
   data/train_transformed/
   data/val_transformed/
   data/test_transformed/

  HANDOVER SUMMARY
  Notebook 03 needs:
  1. models/preprocessing_pipeline  ← fitted pipeline
  2. data/train_transformed/        ← features + label_idx
  3. data/val_transformed/          ← for hyperparameter tuning
  4. data/test_transformed/         ← LOCKED until final eval
